# GDP Prediction Pipeline — Final (Feature engineering + ML tuning + DL)

This notebook is prepared to run in Google Colab. It maintains your previous structure and adds systematic steps to improve accuracy.

**Important:** update `INPUT_CSV` in the configuration cell to point to your cleaned CSV file. Run cells top→down.

In [ ]:

# 0.1 Configuration
import os, math, random, time, re, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import joblib, warnings
warnings.filterwarnings('ignore')
INPUT_CSV = "/content/Base_2011_12.csv"  # change if needed
OUT_DIR = "/content/gdp_pipeline_best_output"
os.makedirs(OUT_DIR, exist_ok=True)
SCALER_TYPE = "standard"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED)
# XGBoost grid options (random sampling)
XGB_PARAM_GRID = {
    "n_estimators": [100, 200, 500],
    "max_depth": [3, 5, 8],
    "eta": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0]
}
XGB_TRIALS = 24
RF_PARAM_GRID = {"n_estimators":[200,500], "max_depth":[5,10,None]}
EPOCHS_DL = 120; PATIENCE = 20; BATCH_SIZE = 8; LEARNING_RATE = 5e-4; LOOKBACK = 5
ACCURACY_PCT_THRESHOLD = 10.0
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, 'INPUT_CSV:', INPUT_CSV, 'OUT_DIR:', OUT_DIR)


In [ ]:

# 0.2 Utilities
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
def rmse(y_true, y_pred): return math.sqrt(mean_squared_error(y_true, y_pred))
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return 100.0 * np.mean(np.abs(y_true - y_pred) / np.where(denom==0, 1, denom))
def metrics_dict(y_true, y_pred, accuracy_pct_threshold=ACCURACY_PCT_THRESHOLD):
    y_true = np.array(y_true); y_pred = np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse_v = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    eps = 1e-9
    mape = np.mean(np.abs((y_true - y_pred) / np.where(np.abs(y_true) < eps, eps, y_true))) * 100
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    smape_v = 100.0 * np.mean(np.abs(y_true - y_pred) / np.where(denom==0, 1, denom))
    perc_errors = np.abs((y_true - y_pred) / np.where(np.abs(y_true) < eps, eps, y_true)) * 100
    accuracy = float(np.mean(perc_errors <= accuracy_pct_threshold) * 100.0)
    return {"MAE": float(mae), "RMSE": float(rmse_v), "R2": float(r2), "MAPE": float(mape), "SMAPE": float(smape_v), "Accuracy": accuracy}
def plot_preds(y_true, y_pred, years=None, title='Actual vs Predicted'):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,4))
    if years is None:
        plt.plot(y_true, label='Actual', marker='o'); plt.plot(y_pred, label='Predicted', marker='x')
    else:
        plt.plot(years, y_true, label='Actual', marker='o'); plt.plot(years, y_pred, label='Predicted', marker='x')
    plt.title(title); plt.legend(); plt.grid(alpha=0.3); plt.show()


In [ ]:

# 1. Load cleaned CSV
import pandas as pd, os
if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")
df = pd.read_csv(INPUT_CSV, header=0)
print('Loaded', df.shape)
display(df.head(3))
if 'Year' not in df.columns:
    raise ValueError('Year column missing') 
df_model = df.copy()


In [ ]:

# 2. Feature engineering (lags, rolling, pct change)
numeric_cols = [c for c in df_model.columns if c!='Year' and pd.api.types.is_numeric_dtype(df_model[c])]
gdp_candidates = [c for c in numeric_cols if ('gross domestic' in c.lower() or 'gdp' in c.lower())]
gdp_const = next((c for c in gdp_candidates if 'constant' in c.lower()), None)
gdp_curr = next((c for c in gdp_candidates if 'current' in c.lower()), None)
TARGET = gdp_const or gdp_curr or (gdp_candidates[-1] if gdp_candidates else numeric_cols[-1])
base_features = [c for c in numeric_cols if c not in (TARGET, gdp_const, gdp_curr)]
print('TARGET=', TARGET, 'base features count=', len(base_features))

lags=[1,2,3]
for c in base_features + [TARGET]:
    for l in lags:
        df_model[f"{c}_lag{l}"] = df_model[c].shift(l)
for c in base_features:
    df_model[f"{c}_rm3"] = df_model[c].rolling(window=3, min_periods=1).mean().shift(1)
    df_model[f"{c}_pctchg1"] = df_model[c].pct_change().shift(1)
df_model = df_model.dropna().reset_index(drop=True)
print('After FE:', df_model.shape)
display(df_model.head(3))


In [ ]:

# 2.1 Feature selection
TARGET_PREF='constant'
gdp_candidates = [c for c in df_model.columns if ('gross domestic' in c.lower() or 'gdp' in c.lower())]
gdp_const = next((c for c in gdp_candidates if 'constant' in c.lower()), None)
gdp_curr = next((c for c in gdp_candidates if 'current' in c.lower()), None)
TARGET = gdp_const if TARGET_PREF=='constant' and gdp_const is not None else (gdp_curr if TARGET_PREF=='current' and gdp_curr is not None else (gdp_const or gdp_curr or gdp_candidates[-1]))
exclude={'Year'} 
if gdp_const is not None: exclude.add(gdp_const)
if gdp_curr is not None: exclude.add(gdp_curr)
FEATURES = [c for c in df_model.columns if c not in exclude]
print('TARGET ->', TARGET); print('Num features ->', len(FEATURES))


In [ ]:

# 3. Sort, split, scale
import re
def year_to_int(s):
    m = re.search(r"(19|20)\d{2}", str(s)); return int(m.group(0)) if m else 0
df_model['_ykey'] = df_model['Year'].apply(year_to_int)
df_model = df_model.sort_values('_ykey').reset_index(drop=True)
N = len(df_model); test_size=max(1,int(0.2*N)); val_size=max(1,int(0.1*(N-test_size)))
train_end = N - test_size - val_size
train_df = df_model.iloc[:train_end].reset_index(drop=True); val_df = df_model.iloc[train_end:train_end+val_size].reset_index(drop=True); test_df = df_model.iloc[train_end+val_size:].reset_index(drop=True)
print('Splits:', len(train_df), len(val_df), len(test_df))
from sklearn.preprocessing import StandardScaler, MinMaxScaler
scaler = StandardScaler() if SCALER_TYPE=='standard' else MinMaxScaler()
X_train = scaler.fit_transform(train_df[FEATURES]); X_val = scaler.transform(val_df[FEATURES]); X_test = scaler.transform(test_df[FEATURES])
y_train = train_df[TARGET].values; y_val = val_df[TARGET].values; y_test = test_df[TARGET].values
joblib.dump(scaler, os.path.join(OUT_DIR,'scaler.joblib'))
print('Saved scaler. X_train shape:', X_train.shape)


In [ ]:

# 4. Sequence builder for DL
import numpy as np, torch
LOOKBACK = int(LOOKBACK)
def create_sequences_array(df_in, features, target, lookback=LOOKBACK):
    arr_X = df_in[features].values; arr_y = df_in[target].values
    Xs, ys = [], []
    for i in range(len(arr_X)-lookback):
        Xs.append(arr_X[i:i+lookback]); ys.append(arr_y[i+lookback])
    return np.array(Xs), np.array(ys)
X_seq, y_seq = create_sequences_array(df_model, FEATURES, TARGET, lookback=LOOKBACK)
train_seq_end = max(0, train_end - LOOKBACK); val_seq_end = max(train_seq_end, train_end + len(val_df) - LOOKBACK)
X_seq_train = X_seq[:train_seq_end]; y_seq_train = y_seq[:train_seq_end]
X_seq_val = X_seq[train_seq_end:val_seq_end]; y_seq_val = y_seq[train_seq_end:val_seq_end]
X_seq_test = X_seq[val_seq_end:]; y_seq_test = y_seq[val_seq_end:]
from torch.utils.data import Dataset, DataLoader
class TimeSeriesDataset(Dataset):
    def __init__(self,X,y): self.X=torch.tensor(X,dtype=torch.float32); self.y=torch.tensor(y,dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self,idx): return self.X[idx], self.y[idx]
train_loader = DataLoader(TimeSeriesDataset(X_seq_train,y_seq_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TimeSeriesDataset(X_seq_val,y_seq_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TimeSeriesDataset(X_seq_test,y_seq_test), batch_size=BATCH_SIZE, shuffle=False)
print('DL dataset shapes:', X_seq_train.shape, X_seq_val.shape, X_seq_test.shape)


In [ ]:

# 5. RandomForest baseline
from sklearn.ensemble import RandomForestRegressor
from itertools import product
best_rf=None; best_rf_score=float('inf'); rf_results=[]
for n_est, depth in product(RF_PARAM_GRID['n_estimators'], RF_PARAM_GRID['max_depth']):
    model = RandomForestRegressor(n_estimators=n_est, max_depth=depth, random_state=RANDOM_SEED, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    m = metrics_dict(y_val, preds)
    rf_results.append((n_est, depth, m))
    if m['RMSE'] < best_rf_score:
        best_rf_score = m['RMSE']; best_rf = model
print('Best RF val RMSE:', best_rf_score)
joblib.dump(best_rf, os.path.join(OUT_DIR,'best_rf.joblib'))


In [ ]:

# 6. XGBoost random-search using xgb.train (robust)
import xgboost as xgb, random, json
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=FEATURES)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=FEATURES)
def sample_params(grid):
    return {'eta': float(random.choice(grid['eta'])), 'max_depth': int(random.choice(grid['max_depth'])),
            'subsample': float(random.choice(grid['subsample'])), 'colsample_bytree': float(random.choice(grid['colsample_bytree']))}
best_bst=None; best_score=float('inf'); trials=[]
for t in range(XGB_TRIALS):
    params = sample_params(XGB_PARAM_GRID); params['objective']='reg:squarederror'; params['seed']=int(RANDOM_SEED)
    num_boost = int(random.choice(XGB_PARAM_GRID['n_estimators']))
    try:
        bst = xgb.train(params, dtrain, num_boost_round=num_boost, evals=[(dtrain,'train'),(dval,'val')], early_stopping_rounds=10, verbose_eval=False)
    except Exception as e:
        bst = xgb.train(params, dtrain, num_boost_round=num_boost, evals=[(dtrain,'train'),(dval,'val')], verbose_eval=False)
    nt = getattr(bst,'best_ntree_limit', None)
    preds_val = bst.predict(dval, ntree_limit=nt) if nt is not None else bst.predict(dval)
    m = metrics_dict(y_val, preds_val)
    trials.append({'params':params,'num_boost':num_boost,'metrics':m,'best_ntree_limit':nt})
    if m['RMSE'] < best_score:
        best_score = m['RMSE']; best_bst = bst
    print(f'Trial {t+1}/{XGB_TRIALS} val RMSE={m["RMSE"]:.4f} MAPE={m["MAPE"]:.2f}')
print('Best XGB val RMSE:', best_score)
if best_bst is not None:
    best_bst.save_model(os.path.join(OUT_DIR,'best_xgb_booster.json'))
    with open(os.path.join(OUT_DIR,'xgb_trials.json'),'w') as f: json.dump(trials,f,indent=2)
    class BoosterWrapper:
        def __init__(self,b,feat): self.b=b; self.feat=feat; self.nt=getattr(b,'best_ntree_limit',None)
        def predict(self,X): d=xgb.DMatrix(X, feature_names=self.feat); return self.b.predict(d, ntree_limit=self.nt) if self.nt is not None else self.b.predict(d)
    best_xgb = BoosterWrapper(best_bst, FEATURES)
else:
    best_xgb=None


In [ ]:

# 6.1 Evaluate ML on test
ml_summary={}
if best_rf is not None:
    preds_rf = best_rf.predict(X_test); ml_summary['RandomForest']=metrics_dict(y_test,preds_rf); print('RF test:', ml_summary['RandomForest'])
if best_xgb is not None:
    preds_xgb = best_xgb.predict(X_test); ml_summary['XGBoost']=metrics_dict(y_test,preds_xgb); print('XGB test:', ml_summary['XGBoost'])
with open(os.path.join(OUT_DIR,'ml_summary.json'),'w') as f: json.dump(ml_summary,f,indent=2)


In [ ]:

# 7. Ensemble & residual correction
weights={}
if 'RandomForest' in ml_summary and 'XGBoost' in ml_summary:
    rf_rmse = ml_summary['RandomForest']['RMSE']; xgb_rmse = ml_summary['XGBoost']['RMSE']
    inv=[1.0/rf_rmse,1.0/xgb_rmse]; total=sum(inv); weights['RandomForest']=inv[0]/total; weights['XGBoost']=inv[1]/total
elif 'RandomForest' in ml_summary:
    weights['RandomForest']=1.0
elif 'XGBoost' in ml_summary:
    weights['XGBoost']=1.0
print('Weights:',weights)
preds_ensemble=None
if 'RandomForest' in ml_summary and 'XGBoost' in ml_summary:
    preds_ensemble = weights['RandomForest']*best_rf.predict(X_test) + weights['XGBoost']*best_xgb.predict(X_test)
elif 'RandomForest' in ml_summary:
    preds_ensemble = best_rf.predict(X_test)
elif 'XGBoost' in ml_summary:
    preds_ensemble = best_xgb.predict(X_test)
if preds_ensemble is not None:
    ensemble_metrics = metrics_dict(y_test,preds_ensemble); print('Ensemble test:', ensemble_metrics); ml_summary['Ensemble']=ensemble_metrics
    from sklearn.linear_model import LinearRegression
    X_tr_val = np.vstack([X_train,X_val]); y_tr_val = np.concatenate([y_train,y_val])
    preds_tr = weights.get('RandomForest',0)*np.concatenate([best_rf.predict(X_train) if best_rf else np.zeros(len(X_train)), best_rf.predict(X_val) if best_rf else np.zeros(len(X_val))]) + weights.get('XGBoost',0)*np.concatenate([best_xgb.predict(X_train) if best_xgb else np.zeros(len(X_train)), best_xgb.predict(X_val) if best_xgb else np.zeros(len(X_val))])
    resid = y_tr_val - preds_tr
    lr_corr = LinearRegression().fit(preds_tr.reshape(-1,1), resid)
    preds_corr = preds_ensemble + lr_corr.predict(preds_ensemble.reshape(-1,1))
    corr_metrics = metrics_dict(y_test, preds_corr); print('Corrected ensemble:', corr_metrics); ml_summary['Ensemble_corrected']=corr_metrics
    pd.DataFrame({'y_true':y_test,'y_ensemble':preds_ensemble,'y_corr':preds_corr}).to_csv(os.path.join(OUT_DIR,'ensemble_preds.csv'),index=False)


In [ ]:

# 8. DL models definitions
import torch, torch.nn as nn
class LSTMForecast(nn.Module):
    def __init__(self, n_features, hidden_size=128, n_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=n_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        out, (h,c) = self.lstm(x); last = out[:, -1, :]; return self.fc(last).squeeze(-1)

class TimeSeriesTransformer(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8, num_encoder_layers=4, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.regressor = nn.Sequential(nn.Linear(d_model,128), nn.ReLU(), nn.Linear(128,64), nn.ReLU(), nn.Linear(64,1))
    def forward(self, x):
        x = self.input_proj(x); out = self.transformer_encoder(x); last = out[:, -1, :]; return self.regressor(last).squeeze(-1)

n_features = len(FEATURES)
print('n_features=', n_features)


In [ ]:

# 9. DL training utilities
from torch.utils.data import DataLoader
def train_one_epoch(model, loader, optimizer, criterion, device=DEVICE):
    model.train(); running=0.0; n=0
    for Xb,yb in loader:
        Xb,yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad(); preds = model(Xb); loss = criterion(preds, yb); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        running += loss.item()*Xb.size(0); n += Xb.size(0)
    return running / max(1,n)

@torch.no_grad()
def evaluate_model(model, loader, criterion, device=DEVICE):
    model.eval(); ys=[]; preds=[]; running=0.0
    for Xb,yb in loader:
        Xb,yb = Xb.to(device), yb.to(device); out = model(Xb); loss = criterion(out,yb); running += loss.item()*Xb.size(0)
        ys.append(yb.detach().cpu().numpy()); preds.append(out.detach().cpu().numpy())
    if len(ys)==0: return None, None, None
    ys = np.concatenate(ys); preds = np.concatenate(preds); return running/(len(ys)+1e-9), ys, preds

def fit_model(model, train_loader, val_loader, epochs=EPOCHS_DL, lr=LEARNING_RATE, patience=PATIENCE):
    criterion = nn.MSELoss(); optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    best_val=float('inf'); best_w=None; no_imp=0; history={'train_loss':[],'val_loss':[]}
    for ep in range(1, epochs+1):
        t0=time.time(); tr = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, yv, pv = evaluate_model(model, val_loader, criterion)
        history['train_loss'].append(tr); history['val_loss'].append(val_loss if val_loss is not None else float('nan'))
        if val_loss is not None and val_loss < best_val - 1e-6:
            best_val = val_loss; best_w = {k:v.cpu().clone() for k,v in model.state_dict().items()}; no_imp=0
        else:
            no_imp += 1
        print(f'Epoch {ep}/{epochs} tr={tr:.6f} val={val_loss:.6f} time={(time.time()-t0):.1f}s')
        if no_imp >= patience: print('Early stopping'); break
    if best_w is not None: model.load_state_dict(best_w)
    return model, history


In [ ]:

# 10. Train LSTM
import torch
lstm = LSTMForecast(n_features=n_features, hidden_size=128, n_layers=2).to(DEVICE)
lstm, lstm_hist = fit_model(lstm, train_loader, val_loader, epochs=EPOCHS_DL, lr=LEARNING_RATE, patience=PATIENCE)
_, y_test_lstm, y_pred_lstm = evaluate_model(lstm, test_loader, nn.MSELoss())
lstm_metrics_test = metrics_dict(y_test_lstm, y_pred_lstm) if y_test_lstm is not None else None
print('LSTM test metrics:', lstm_metrics_test)
torch.save(lstm.state_dict(), os.path.join(OUT_DIR,'lstm_weights.pt'))


In [ ]:

# 11. Train Transformer
trans = TimeSeriesTransformer(n_features=n_features, d_model=128, n_heads=8, num_encoder_layers=4).to(DEVICE)
trans, trans_hist = fit_model(trans, train_loader, val_loader, epochs=EPOCHS_DL, lr=LEARNING_RATE, patience=PATIENCE)
_, y_test_trans, y_pred_trans = evaluate_model(trans, test_loader, nn.MSELoss())
transformer_metrics_test = metrics_dict(y_test_trans, y_pred_trans) if y_test_trans is not None else None
print('Transformer test metrics:', transformer_metrics_test)
torch.save(trans.state_dict(), os.path.join(OUT_DIR,'transformer_weights.pt'))


In [ ]:

# 12. Final comparison & save
results={}
if 'ml_summary' in globals():
    for k,v in ml_summary.items(): results[k]=v
if 'lstm_metrics_test' in globals() and lstm_metrics_test is not None: results['LSTM']=lstm_metrics_test
if 'transformer_metrics_test' in globals() and transformer_metrics_test is not None: results['Transformer']=transformer_metrics_test
if len(results)==0: print('No results yet. Run model cells.')
else:
    res_df = pd.DataFrame(results).T
    cols=[c for c in ['MAE','RMSE','R2','MAPE','SMAPE','Accuracy'] if c in res_df.columns]
    res_df = res_df[cols]; display(res_df)
    res_df.to_csv(os.path.join(OUT_DIR,'model_comparison_results_final.csv'), index=True)
    print('Saved final results to', os.path.join(OUT_DIR,'model_comparison_results_final.csv'))
